<a href="https://colab.research.google.com/github/katelynnlindsey/phonology-chuvash/blob/main/logbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Acoustic analysis of Chuvash vowels

1. Obtain acoustic data

Chuvash-Voice

In [ ]:
from datasets import load_dataset, DatasetDict, concatenate_datasets, Audio

chuvash_voice = DatasetDict()
chuvash_voice = load_dataset("alexantonov/chuvash_voice")
chuvash_voice = chuvash_voice.cast_column("audio", Audio(sampling_rate=16000))

Common-Voice

In [ ]:
https://datacollective.mozillafoundation.org/datasets/cmj8u3oz40059nxxbu0wyqk1t

Phonology class

2. Pre-process for Montreal Forced Aligner (MFA)

3. Run MFA

4. Pre-process for FAVE vowel extraction

chuvash-manual-recode.py

In [ ]:
import textgrid
import os
import codecs
import tempfile
import re
from pydub import AudioSegment
import shutil
import sys # <--- ADDED THIS IMPORT

# Define your IPA to ARPAbet mapping
ipa_to_arpabet_map = {
    'ʌ': 'AH', 'ɑ': 'AA', 'u': 'UW', 'y': 'UX', 'ɛ': 'EH', 'e': 'EY', 'o': 'OW', 'i': 'IY', 'ɯ': 'IX',
    'sil': 'SIL', 'v': 'V', 'j': 'Y', 'n': 'N', 'ʃ': 'SH', 'p': 'P', 'k': 'K', 'm': 'M', 'r': 'R',
    # Add any other IPA consonants you have that need specific ARPAbet mapping
}

# --- TextGrid Processing Function (unchanged for core logic, just verbosity) ---
def process_single_textgrid(input_tg_path, output_tg_path, mapping):
    temp_filepath = None
    try:
        content = None
        encodings_to_try = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']

        for enc in encodings_to_try:
            try:
                with codecs.open(input_tg_path, 'r', encoding=enc) as f:
                    content = f.read()
                    if "File type" in content and "Object class" in content and ("text =" in content or "intervals:" in content):
                        break
                    else:
                        content = None
            except UnicodeDecodeError:
                content = None
            except Exception as e:
                print(f"Unexpected error during encoding trial for TG '{os.path.basename(input_tg_path)}' with {enc}: {e}", file=sys.stderr)
                content = None

        if not content:
            print(f"Warning: Could not reliably read TG '{os.path.basename(input_tg_path)}' with common encodings. Skipping TextGrid processing.", file=sys.stderr)
            return

        with tempfile.NamedTemporaryFile(mode='w', delete=False, encoding='utf-8', suffix='.TextGrid', newline='') as temp_file:
            temp_file.write(content)
            temp_filepath = temp_file.name

        try:
            tg = textgrid.TextGrid.fromFile(temp_filepath)
        except Exception as e:
            print(f"Error: textgrid.TextGrid.fromFile() failed to parse temp TG '{os.path.basename(input_tg_path)}' ({temp_filepath}): {e}. Skipping TextGrid processing.", file=sys.stderr)
            return

        phones_tier = tg.getFirst('phones')

        if phones_tier:
            for interval in phones_tier:
                original_label = interval.mark.strip()
                recoded_label = mapping.get(original_label, original_label)
                interval.mark = recoded_label
            # print(f"Recoded 'phones' tier in '{os.path.basename(input_tg_path)}'.") # Mute for less verbose batch output
        else:
            print(f"Warning: 'phones' tier not found in '{os.path.basename(input_tg_path)}'. No recoding performed for this TG.", file=sys.stderr)

        tg.write(output_tg_path)
        # print(f"Saved recoded TextGrid to '{os.path.basename(output_tg_path)}'.") # Mute for less verbose batch output

    except Exception as e:
        print(f"An unexpected error occurred during TG processing '{os.path.basename(input_tg_path)}': {e}", file=sys.stderr)
    finally:
        if temp_filepath and os.path.exists(temp_filepath):
            os.remove(temp_filepath)

# --- Audio Processing Function (NEW) ---
def convert_or_copy_audio(input_audio_path, output_wav_path):
    base_name = os.path.basename(input_audio_path)
    output_base_name = os.path.basename(output_wav_path)

    if input_audio_path.lower().endswith(".mp3"):
        try:
            audio = AudioSegment.from_mp3(input_audio_path)
            # Ensure 16-bit, 1 channel (mono), 16000 Hz - standard for speech tools
            audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)
            audio.export(output_wav_path, format="wav")
            print(f"Converted '{base_name}' to '{output_base_name}'.")
            return True
        except Exception as e:
            print(f"Error converting MP3 '{base_name}' to WAV: {e}. Skipping audio conversion.", file=sys.stderr)
            return False
    elif input_audio_path.lower().endswith(".wav"):
        try:
            shutil.copy(input_audio_path, output_wav_path)
            # print(f"Copied existing WAV '{base_name}' to '{output_base_name}'.") # Mute for less verbose batch output
            return True
        except Exception as e:
            print(f"Error copying WAV '{base_name}': {e}. Skipping audio copy.", file=sys.stderr)
            return False
    else:
        print(f"Warning: Unsupported audio format for '{base_name}'. Skipping audio processing.", file=sys.stderr)
        return False

# --- Main Directory Processing Logic ---
base_corpus_path = "C:\\Users\\profk\\Documents\\GitHub\\phonology-chuvash\\corpora"
input_dir = os.path.join(base_corpus_path, "subcorpus")
output_recoded_tg_dir = os.path.join(base_corpus_path, "recoded_textgrids")
output_converted_audio_dir = os.path.join(base_corpus_path, "converted_audio") # New directory for prepared WAVs

# Create output directories if they don't exist
os.makedirs(output_recoded_tg_dir, exist_ok=True)
os.makedirs(output_converted_audio_dir, exist_ok=True)

print(f"Starting preparation process for files in: {input_dir}")
print(f"Recoded TextGrids will be saved to: {output_recoded_tg_dir}")
print(f"Converted/Copied audio will be saved to: {output_converted_audio_dir}")

processed_tgs_count = 0
processed_audio_count = 0
# Use sets to store base names of files that need processing or have been processed
all_basenames = set() # All unique base names (e.g., 'utterance_000000') found

# First pass: Identify all relevant base names (from TextGrids, MP3s, WAVs)
for filename in os.listdir(input_dir):
    name, ext = os.path.splitext(filename)
    if ext.lower() in [".textgrid", ".mp3", ".wav"]:
        all_basenames.add(name)

print(f"\nFound {len(all_basenames)} unique base names to process.")

# Second pass: Process TextGrids for each identified basename
print("\n--- Processing TextGrid files ---")
for base_name in sorted(list(all_basenames)): # Process in sorted order for consistency
    input_tg_path = os.path.join(input_dir, f"{base_name}.TextGrid")
    output_tg_path = os.path.join(output_recoded_tg_dir, f"{base_name}_arpabet.TextGrid")

    if not os.path.exists(input_tg_path):
        print(f"Warning: No TextGrid found for '{base_name}'. Skipping TextGrid recoding.", file=sys.stderr)
        continue # Skip to next basename if no TextGrid

    # print(f"Preparing TextGrid: {base_name}.TextGrid") # Muted for less verbose output
    process_single_textgrid(input_tg_path, output_tg_path, ipa_to_arpabet_map)
    processed_tgs_count += 1

print(f"\nFinished recoding {processed_tgs_count} TextGrids.")
print(f"Recoded TextGrids are in: {output_recoded_tg_dir}")

# Third pass: Process Audio Files (MP3 to WAV conversion / WAV copy) for each identified basename
print("\n--- Processing Audio files (MP3 to WAV conversion / WAV copy) ---")
for base_name in sorted(list(all_basenames)): # Process in sorted order for consistency
    original_mp3_path = os.path.join(input_dir, f"{base_name}.mp3")
    original_wav_path = os.path.join(input_dir, f"{base_name}.wav")

    target_output_wav_path = os.path.join(output_converted_audio_dir, f"{base_name}.wav")

    # Skip if WAV already exists in the converted_audio dir
    if os.path.exists(target_output_wav_path):
        # print(f"Skipping audio for '{base_name}': '{os.path.basename(target_output_wav_path)}' already exists in destination.") # Mute for less verbose output
        processed_audio_count += 1
        continue

    # Prioritize MP3 conversion if MP3 exists
    if os.path.exists(original_mp3_path):
        # print(f"Converting audio: {base_name}.mp3") # Muted for less verbose output
        if convert_or_copy_audio(original_mp3_path, target_output_wav_path):
            processed_audio_count += 1
    elif os.path.exists(original_wav_path):
        # print(f"Copying audio: {base_name}.wav") # Muted for less verbose output
        if convert_or_copy_audio(original_wav_path, target_output_wav_path):
            processed_audio_count += 1
    else:
        print(f"Warning: No MP3 or WAV audio found for base name '{base_name}'. Skipping audio processing.", file=sys.stderr)

print(f"\nFinished audio preparation for {processed_audio_count} files.")
print("\nmanual_recode_directory.py script finished.")
print(f"Prepared WAVs are in: {output_converted_audio_dir}")

5. Run new-FAVE

In [ ]:
!pip install new-fave

In [ ]:
chuvash-fave.py

In [ ]:
from new_fave import fave_audio_textgrid, write_data
import os
import sys # For error output

# --- Directory Paths ---
base_corpus_path = "C:\\Users\\profk\\Documents\\GitHub\\phonology-chuvash\\corpora"
audio_input_dir = os.path.join(base_corpus_path, "converted_audio")   # NEW: Look for WAVs here
textgrid_input_dir = os.path.join(base_corpus_path, "recoded_textgrids") # Recoded TextGrids
output_results_dir = os.path.join(base_corpus_path, "fave_output")    # Final fave-extract output

# Create output directory if it doesn't exist
os.makedirs(output_results_dir, exist_ok=True)

print(f"Starting fave-extract processing for TextGrids in: {textgrid_input_dir}")
print(f"Audio files from: {audio_input_dir}")
print(f"Results will be saved to: {output_results_dir}")

processed_count = 0

# Collect successful TextGrid/Audio pairs to process
files_to_process = []
for filename in os.listdir(textgrid_input_dir):
    if filename.endswith("_arpabet.TextGrid"):
        recoded_tg_path = os.path.join(textgrid_input_dir, filename)
        base_name = filename.replace("_arpabet.TextGrid", "")
        audio_path = os.path.join(audio_input_dir, f"{base_name}.wav")

        if not os.path.exists(audio_path):
            print(f"Warning: Prepared audio file '{os.path.basename(audio_path)}' not found in '{audio_input_dir}' for TextGrid '{filename}'. Skipping this pair.", file=sys.stderr)
            continue

        files_to_process.append((audio_path, recoded_tg_path, base_name))

if not files_to_process:
    print("\nNo valid audio-TextGrid pairs found for fave-extract. Ensure preparation script ran successfully.", file=sys.stderr)
    sys.exit(0) # Exit cleanly if nothing to process

print(f"\nFound {len(files_to_process)} audio-TextGrid pairs to process.")

for audio_path, recoded_tg_path, base_name in files_to_process:
    print(f"Processing: {base_name}...")
    try:
        speakers_output = fave_audio_textgrid(
            audio_path = audio_path,
            textgrid_path = recoded_tg_path,
            speakers = "all",
            labelset_parser = "cmu_parser",
            point_heuristic = "fave",
            ft_config = "default"
        )

        write_data(
            speakers_output,
            destination = output_results_dir,
            separate=False
        )
        # print(f"Successfully processed and wrote data for '{base_name}'.") # Mute for less verbose output during batch
        processed_count += 1

    except Exception as e:
        print(f"Error processing '{base_name}': {e}. Skipping to next file.", file=sys.stderr)
        continue

print(f"\nFinished processing {processed_count} audio-TextGrid pairs.")
print(f"All fave-extract results saved to: {output_results_dir}")
print("\nchuvash_fave_directory.py script finished.")

6. Combine extracted vowel data

chuvash-combine.py

In [ ]:
import pandas as pd
import os
import sys

# --- Define Paths ---
base_corpus_path = "C:\\Users\\profk\\Documents\\GitHub\\phonology-chuvash\\corpora"
fave_output_dir = os.path.join(base_corpus_path, "fave_output")
combined_csv_path = os.path.join(base_corpus_path, "all_chuvash_vowel_points.csv") # The master CSV

print(f"Starting data combination process from: {fave_output_dir}")
print(f"Combined data will be saved to: {combined_csv_path}")

all_points_dfs = []
processed_files_count = 0
skipped_files_count = 0

# Iterate through all files in the fave_output directory
for filename in os.listdir(fave_output_dir):
    # We are specifically interested in the '_points.csv' files
    if filename.endswith("_points.csv"):
        file_path = os.path.join(fave_output_dir, filename)

        try:
            # Read the CSV, explicitly specifying UTF-8 encoding for Cyrillic characters.
            # on_bad_lines='warn' will issue a warning and skip malformed lines,
            # which is good for large datasets where a few bad lines shouldn't halt everything.
            df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='warn')
            all_points_dfs.append(df)
            processed_files_count += 1
            # Optional: print(f"Loaded: {filename}") # Uncomment to see each file being loaded
        except pd.errors.EmptyDataError:
            print(f"Warning: {filename} is empty. Skipping.", file=sys.stderr)
            skipped_files_count += 1
        except Exception as e:
            # Catch other potential errors during file reading
            print(f"Error reading {filename}: {e}. Skipping.", file=sys.stderr)
            skipped_files_count += 1

print(f"\nFinished attempting to load files. Successfully loaded {processed_files_count} files, skipped {skipped_files_count} files.")

if all_points_dfs:
    print(f"Concatenating {len(all_points_dfs)} DataFrames...")
    combined_df = pd.concat(all_points_dfs, ignore_index=True)
    print(f"Combined DataFrame created with {len(combined_df)} entries.")

    # Save the combined DataFrame to a new CSV file
    combined_df.to_csv(combined_csv_path, index=False, encoding='utf-8')
    print(f"Combined data successfully saved to: {combined_csv_path}")

    # Optional: Display some info about the combined data
    # print("\nCombined DataFrame Info:")
    # combined_df.info()
    # print("\nCombined DataFrame Head (first 5 rows):")
    # print(combined_df.head())

else:
    print("No '_points.csv' files were successfully loaded. Cannot create combined DataFrame.")

print("\nData combination script finished.")

7. Visualize results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Define Path to the COMBINED CSV ---
base_corpus_path = "C:\\Users\\profk\\Documents\\GitHub\\phonology-chuvash\\corpora"
combined_csv_path = os.path.join(base_corpus_path, "all_chuvash_vowel_points.csv")

# --- Load the combined data ONCE ---
try:
    combined_df = pd.read_csv(combined_csv_path, encoding='utf-8')
    print(f"Successfully loaded combined data with {len(combined_df)} entries.")
    print("Columns available:", combined_df.columns.tolist())

    # Filter for only vowel labels (exclude SIL and any non-vowel markers)
    # Adjust this list if you have other non-vowel labels you want to exclude
    vowels_df = combined_df[~combined_df['label'].isin(['SIL', '<eps>', '#'])].copy()
    print(f"Filtered for vowels: {len(vowels_df)} entries remaining.")

except FileNotFoundError:
    print(f"Error: Combined CSV file not found at '{combined_csv_path}'. Please run 'combine_fave_points_data.py' first.")
    # Exit or handle the error appropriately if running as a script
    # For a notebook, you might just display an error and let the user fix it.
except Exception as e:
    print(f"An error occurred while loading or preparing data: {e}")
    # Handle error
    vowels_df = pd.DataFrame() # Create an empty DataFrame to prevent further errors if loading fails

In [ ]:
# Cell 1: Basic Vowel Plot (F2 vs F1)
if not vowels_df.empty:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=vowels_df, x='F2', y='F1', hue='label', alpha=0.6, s=20)
    plt.gca().invert_xaxis() # Invert F2 axis as is common in phonetics
    plt.gca().invert_yaxis() # Invert F1 axis as is common in phonetics
    plt.title('Chuvash Vowel Plot (F2 vs F1)')
    plt.xlabel('F2 (Hz)')
    plt.ylabel('F1 (Hz)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(title='Vowel Label', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No vowel data available for F1/F2 plot.")

In [ ]:
# Cell 2: F1 Distribution by Vowel
if not vowels_df.empty:
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=vowels_df, x='label', y='F1', inner='quartile', palette='viridis')
    plt.title('F1 Distribution by Vowel')
    plt.xlabel('Vowel Label')
    plt.ylabel('F1 (Hz)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No vowel data available for F1 distribution plot.")

In [ ]:
# Cell 3: Duration Distribution by Vowel
if not vowels_df.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=vowels_df, x='label', y='dur', palette='plasma')
    plt.title('Vowel Duration Distribution')
    plt.xlabel('Vowel Label')
    plt.ylabel('Duration (s)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No vowel data available for duration plot.")